In [ ]:
from huggingface_hub import login
import os
from dotenv import load_dotenv

# os.environ["TRANSFORMERS_CACHE"] = "/content/drive/Shareddrives/Algoverse_KSAC/hf_cache" #stores model
os.environ["HF_HOME"] = "./hf_home"  # stores logins

hf_token = os.getenv('HF_TOKEN')
# print(type(hf_token))
login(token=hf_token)

In [ ]:
import os
from transformers import AutoTokenizer, AutoModelForCausalLM
from pathlib import Path

variants = ['small', 'base', 'large']

paths = [f'./models/google/flan-t5-{variant}' for variant in variants]

save_path = Path('choose_which_path').resolve()

tokenizer = AutoTokenizer.from_pretrained(
    save_path,
    local_files_only=True
)

model = AutoModelForCausalLM.from_pretrained(
    save_path,
    torch_dtype="auto",
    device_map="auto",
    local_files_only=True
)

print("Reloaded model successfully")
print(f"model.device = {model.device}")

#Dataloading (TODO)
#after sparqlgen + name parsing

In [ ]:
import pandas as pd

In [ ]:
names = ['mintaka', 'hotpot', 'qald']

preloading = {name:f'./SPARQL/sparql_{name}' for name in names}

dataframes= {}
for name, path in preloading.items():
    df = pd.read_csv(path)
    dataframes[name] = df




In [ ]:
# from transformers import pipeline
from transformers.generation.utils import GenerationMixin

In [ ]:
# Inference on already-loaded model

def answer(question, context):
    messages = [
        {
            "role": "system",
            "content": (

                """
                You are a precise question-answering engine designed to extract information strictly from provided context.

                CORE PRINCIPLES:
                - Answer questions using ONLY the information explicitly stated in the given context

                RESPONSE GUIDELINES:
                - Begin each answer by restating the question in statement form
                - If the context is ambiguous or incomplete, acknowledge the limitations
                - Never infer, extrapolate, or add information beyond what's provided
        

                FORMAT EXAMPLE:
                Question: "What is the capital of France?"
                Context: "Paris"
                Answer: "The capital of France is Paris."

                Question: "What is the tallest mountain in the world?"
                Context: "Mount Everest"
                Answer: "The tallest mountain in the world is Mount Everest."

                - Lead with the restated question as a statement
                - Follow with the answer derived from context
                - Keep responses focused and avoid unnecessary elaboration




                """
            ),
        },
        {
            "role": "user",
            "content": f"Question: {question} Context: {context} Answer: "  #Joining a list into a string
        }
    ]

    inputs = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=True,
        return_dict=True,
        return_tensors="pt",
        enable_thinking=False
    ).to(model.device)

    outputs = model.generate(**inputs, max_new_tokens=250, do_sample=False, temperature=0)  #Increasing the temperature, and only considering tokens that make up 90% of probability (p = prob)

    # dummy = outputs[0][inputs["input_ids"].shape[-1]:]

    output_answer = tokenizer.decode(
        outputs[0][inputs["input_ids"].shape[-1]:],  # Slicing the length of the input tokens to get what comes after (Getting the length) (start:end)
        skip_special_tokens=True
    ).split('\n')

    # breakpoint()
    # import pdb; pdb.set_trace()
    return output_answer


answered = {}



for name, data in dataframes.items():
  answers = []
  for index, row in data.iterrows():
      question = row["SAE Question"]
      context = row['Names']
      llmanswer = answer(question,context)
      
      print(f"Processed row {index + 1} question")



  data['Answer'] = answers
  answered[name] = data
  print(f'{name} successfully processed', len(answered[name]))

#Output Translation Path





In [ ]:
import ast
import re

def clean_text(text):
    # If the entry is actually a list, flatten it
    if isinstance(text, list):
        text = text[0]
    elif isinstance(text, str):
       #If looks like a list []
        if text.strip().startswith('[') and text.strip().endswith(']'):
            try:
                parsed = ast.literal_eval(text)
                if isinstance(parsed, list) and len(parsed) > 0:
                    text = parsed[0]
            except Exception:
                pass

    # Now clean as before
    if not isinstance(text, str):
        return text

    text = re.sub(r'^(AAVE|SAE)\s*Question[:\s-]*', '', text, flags=re.IGNORECASE)
    text = re.sub(r'^[\s\[\]\'"]+|[\s\[\]\'"]+$', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text



for name, data in answered.items():
# Apply to both columns
  data['SAE Question'] = data['SAE Question'].apply(clean_text)
  data['AAVE Question'] = data['AAVE Question'].apply(clean_text)
  file_path = f'./LLM_answers/LLM_Answers_{name}.csv'
  data.to_csv(file_path, index=False)
  print(f"Translations completed and saved to {file_path}")



